In [ ]:
# import json

# # Load the original JSON (assuming you have the full file)
# with open("meal_data_2500-2900.json", "r") as f:
#     data = json.load(f)

# # Split into 5 files
# keys = ["2500", "2600", "2700", "2800", "2900"]
# for key in keys:
#     # Create a dictionary with just this key and its list
#     subset = {key: data[key]}
#     # Write to a new file
#     with open(f"meal_data_{key}.json", "w") as f:
#         json.dump(subset, f, indent=4)

In [ ]:
!pip install -q accelerate peft bitsandbytes transformers trl googletrans triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import json
from googletrans import Translator
import random
from tqdm import tqdm
import time
# import os
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, HfArgumentParser, pipeline, logging
# from peft import LoraConfig, PeftModel
# from trl import SFTTrainer

In [ ]:
class MealPlanDataset():
    def __init__(self):
        self.translator = Translator()
        self.translation_dict = {
            "Breakfast": "Bữa sáng",
            "Lunch": "Bữa trưa",
            "Dinner": "Bữa tối",
            "serving": "phần",
            "and": "và",
            "with": "với",
            "&": "và",
            ",": ",",
        }
        self.ingredients = {
            "chicken": "thịt gà",
            "beef": "thịt bò",
            "pork": "thịt heo",
            "salmon": "cá hồi",
            "shrimp": "tôm",
            "crab": "cua",
            "vegetable": "rau",
            "veggie": "rau củ",
            "yogurt": "sữa chua",
            "milk": "sữa",
            "cheese": "phô mai",
            "egg": "trứng",
            "tofu": "đậu phụ",
            "mushroom": "nấm",
            "potato": "khoai tây",
            "rice": "gạo",
            "noodle": "mì",
            "bread": "bánh mì",
            "fish": "cá",
            "pasta": "mì ống",
            "spinach": "rau chân vịt",
            "tomato": "cà chua",
            "carrot": "cà rốt",
            "broccoli": "bông cải xanh",
            "apple": "táo",
            "banana": "chuối",
            "orange": "cam",
            "strawberry": "dâu tây",
            "blueberry": "việt quất",
            "avocado": "bơ",
        }

    def clean_text(self, text):
        """Làm sạch văn bản trước khi dịch"""
        return text.strip().lower()

    def translate_text(self, text):
        """Dịch văn bản với cache và xử lý lỗi"""
        text = self.clean_text(text)
        if text in self.translation_dict:
            return self.translation_dict[text]
        time.sleep(0.5)

        try:
            translated = self.translator.translate(text, src="en", dest="vi").text
            return translated.lower().strip()
        except Exception as e:
            print(f"Lỗi khi dịch '{text}': {e}")
            return text

    def extract_and_translate_foods(self, file_path):
        """Lọc và dịch các cụm món ăn từ dữ liệu JSON"""
        food_set = set()

        # Duyệt qua tất cả các tệp JSON trong thư mục
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Duyệt qua từng mục trong dữ liệu
        for calorie_range, meal_plans in data.items():
            for plan in meal_plans:
                for meal_type, items in plan.items():
                    for item in items:
                        food_set.add(item["food"])

        # Dịch các cụm món ăn và thêm vào translation_dict
        for food in sorted(food_set):
            translated_food = self.translate_text(food)
            self.translation_dict[food] = translated_food
            print(f"{food} -> {translated_food}")

    def analyze_ingredients(self, food_name):
        """Phân tích thành phần từ tên món ăn"""
        detected = []
        separators = [" and ", " with ", " & ", ", "]  # Các dấu phân cách phổ biến
        for sep in separators:
            if sep in food_name:
                parts = food_name.split(sep)
                for part in parts:
                    part = part.strip().lower()
                    for eng, vi in self.ingredients.items():
                        if eng in part:
                            detected.append(vi)
                return detected if detected else ["đa dạng"]

        # Nếu không phải từ ghép, kiểm tra toàn bộ
        food_lower = food_name.lower()
        for eng, vi in self.ingredients.items():
            if eng in food_lower:
                detected.append(vi)
        return detected if detected else ["đa dạng"]

    def generate_prompt_from_meal(self, calories, meal_plan):
        """Tạo prompt dựa trên thực đơn cụ thể"""
        all_ingredients = []
        for items in meal_plan.values():
            for item in items:
                all_ingredients.extend(self.analyze_ingredients(item["food"].lower()))

        main_ingredients = random.sample(
            list(set(all_ingredients)), min(2, len(set(all_ingredients)))
        )

        templates = [
            "Tôi muốn thực đơn khoảng {} calo với các món chứa {}.",
            "Gợi ý thực đơn {} calo có các món chính là {}.",
            "Hãy đề xuất thực đơn khoảng {} calo. Tôi thích ăn {}",
            "Thực đơn {} calo với thành phần chính là {}.",
            "Tôi cần thực đơn {} calo/ngày, ưu tiên món có {}.",
        ]

        prompt = random.choice(templates).format(
            calories, " và ".join(main_ingredients)
        )

        return prompt

    def process_data(self, input_file, output_file):
        """Xử lý dữ liệu từ file JSON và tạo hội thoại"""
        with open(input_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        transformed_data = []

        for calories, meal_plans in tqdm(data.items(), desc="Xử lý"):
            for plan in meal_plans:
                if not plan:  # Bỏ qua các phần tử rỗng
                    continue
                translated_plan = {}
                for meal_type, items in plan.items():
                    translated_plan[self.translate_text(meal_type)] = [
                    {
                        "food": self.translate_text(item["food"]).lower(),
                        "serving": self.translate_text(item["serving"]).lower(),
                    }
                    for item in items
                ]

                prompt = self.generate_prompt_from_meal(calories, plan)

                response = "Thực đơn gợi ý:\n"
                for meal_type, items in translated_plan.items():
                    response += f"- {meal_type}: "
                    response += ", ".join(
                        [f"{item['food']} ({item['serving']})" for item in items]
                    )
                    response += "\n"

                conversation = f"""<|im_start|>system
Bạn là chuyên gia dinh dưỡng AI<|im_end|>
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response.strip()}<|im_end|>"""

                transformed_data.append({'text': ''.join(conversation)})
        return transformed_data

In [ ]:
class DataPreprocessor:
    def __init__(self):
        self.dataset = []

    def load_and_process(self, json_files):
        """Tải và xử lý tất cả file JSON"""
        for file in tqdm(json_files, desc="Processing files"):
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            processor = MealPlanDataset()
            processor.extract_and_translate_foods(file)
            processed = processor.process_data(file, None)  # Không lưu file tạm
            self.dataset.extend(processed)

        return self.dataset

    def save_to_jsonl(self, output_file):
        """Lưu dữ liệu đã xử lý dưới dạng JSONL"""
        with open(output_file, 'w', encoding='utf-8') as f:
            for item in self.dataset:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
# json_files = ['/kaggle/input/raw-meal-datasets/meal_data_1500-1900.json',
#             '/kaggle/input/raw-meal-datasets/meal_data_1000-1400.json',
#               '/kaggle/input/raw-meal-datasets/meal_data_2000-2400.json',
#               '/kaggle/input/raw-meal-datasets/meal_data_2500-2900.json',
#               '/kaggle/input/raw-meal-datasets/meal_data_3000-3500.json']
num = 3500
json_files = [f'/content/drive/MyDrive/meal_data_{num}.json']

preprocessor = DataPreprocessor()
dataset = preprocessor.load_and_process(json_files)
print("Số dòng dữ liệu đã xử lý:", len(dataset))
print("Dòng đầu tiên:")
print(dataset[0] if dataset else "Không có dữ liệu.")
# lưu file vào drive
preprocessor.save_to_jsonl(f'/content/drive/MyDrive/colab/processed_data_{num}.jsonl')

Processing files:   0%|          | 0/1 [00:00<?, ?it/s]

Lỗi khi dịch '"banana cream pie" oatmeal': 'coroutine' object has no attribute 'text'
"Banana Cream Pie" Oatmeal -> "banana cream pie" oatmeal


<ipython-input-11-350f93a2a9e5>:63: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  return text


Lỗi khi dịch '1000 calorie shake': 'coroutine' object has no attribute 'text'
1000 Calorie Shake -> 1000 calorie shake
Lỗi khi dịch '3 bean salad': 'coroutine' object has no attribute 'text'
3 Bean Salad -> 3 bean salad
Lỗi khi dịch '5-minute healthy strawberry frozen yogurt': 'coroutine' object has no attribute 'text'
5-Minute Healthy Strawberry Frozen Yogurt -> 5-minute healthy strawberry frozen yogurt
Lỗi khi dịch 'all american tuna': 'coroutine' object has no attribute 'text'
All American Tuna -> all american tuna
Lỗi khi dịch 'almond butter banana toast': 'coroutine' object has no attribute 'text'
Almond Butter Banana Toast -> almond butter banana toast
Lỗi khi dịch 'almonds': 'coroutine' object has no attribute 'text'
Almonds -> almonds
Lỗi khi dịch 'almonds and blueberries yogurt snack': 'coroutine' object has no attribute 'text'
Almonds and Blueberries Yogurt Snack -> almonds and blueberries yogurt snack
Lỗi khi dịch 'ants on a log': 'coroutine' object has no attribute 'text'
A


Xử lý:   0%|          | 0/1 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Lỗi khi dịch '1 serving': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'no-bake apple "cookies"': 'coroutine' object has no attribute 'text'
Lỗi khi dịch '1 slice': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'breakfast': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'strawberry grape granola parfait': 'coroutine' object has no attribute 'text'
Lỗi khi dịch '1 serving': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'tuna apple salad': 'coroutine' object has no attribute 'text'
Lỗi khi dịch '1 serving': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'lunch': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'dragon noodles with crab': 'coroutine' object has no attribute 'text'
Lỗi khi dịch '3 servings': 'coroutine' object has no attribute 'text'
Lỗi khi dịch 'avocado': 'coroutine' object has no attribute 'text'
Lỗi khi dịch '1 avocado': 'coroutine' object has no attribute 'text'
